In [ ]:
import seaborn             as sns
import matplotlib.pyplot   as plt
import numpy               as np
import json
import os

from pymatgen.core.structure import Structure
from pymatgen.io.vasp.inputs import Poscar

from libraries import adsorption, energy_lookup, vasp_files

sns.set_theme()


# Hydrogen adsorption on a slab

For the selected (relaxed) surface this notebook samples **atop, bridge and hollow** sites of the top layer with a periodic
Voronoi decomposition, places an H atom or an H$_2$ molecule on each inequivalent site, relaxes every configuration with the
ML-IAP at fixed cell, and computes the adsorption free energy of each configuration (the lowest one of each adsorbate is reported):

\begin{equation}
    \Delta G_{ads} = G_{conf} - \left( n G_{surf} + \tfrac{n_H}{2} G_{H_2} \right)
\end{equation}

per H atom (adsorbate H, $n_H = 1$ for the configurations built here) or per H$_2$ molecule (adsorbate H$_2$, $n_H = 2$), where $n$
is the number of surface cells of the configuration.

The first part of the notebook evaluates the free energies at the **0 K** level, $G = E_0$ (the total energy of each relaxed structure), so
$\Delta G_{ads}$ is the adsorption energy printed and saved below. The last section (optional) evaluates it at a temperature,
$G(T) = U_0(T) + F_{vib}(T)$, from NPT runs of the surface and of every configuration.

All free energies (surface, configurations and the isolated H$_2$ reference) must come from the same level of theory and from the same
level of the free energy.

In [ ]:
# Define the slab surface and create the H-adsorption workspace beneath it.
general_folder = 'input/test/slab_v1'
surface = '0_1_0_i_3'

# Which termination of the slab the adsorbate goes on ('top' or 'bottom'; they differ for asymmetric slabs)
surface_side = 'top'

# ML-IAP used for every energy that is not read from a DFT calculation (vasprun.xml)
model_load_path = 'mace-mpa-0-medium.model'
device          = 'cpu'    # switch to 'cuda' if a GPU is available

# An 'H2' configuration whose relaxed H-H distance exceeds this (Å) has dissociated (the molecule is ~0.75 Å)
hh_molecular_threshold = 1.1

surface_dir = f'{general_folder}/{surface}'
path_to_confs = f'{surface_dir}/H-absorption'
path_to_free_H = f'{path_to_confs}/free-H'
path_to_free_H2 = f'{path_to_confs}/free-H2'
path_to_H2_gas = f'{path_to_confs}/H2-gas'  # isolated H2 molecule reference

os.makedirs(path_to_confs, exist_ok=True)
print(f'Using surface folder: {surface_dir}')
print(f'Generating adsorption samples in: {path_to_confs}')

## Reference structures
The bare surface and the isolated H$_2$ molecule are relaxed first, because the adsorption sites are built on the relaxed surface
(`relaxation/CONTCAR`). Relaxations keep the cell fixed so that all configurations share the lattice vectors of the surface.
Failures raise an error instead of being skipped silently.

In [ ]:
# Isolated H2 molecule in a large box
box = 15.0
h2_gas = Structure(np.eye(3)*box, ['H', 'H'], [[0.0, 0.0, 0.0], [0.0, 0.0, 0.74]], coords_are_cartesian=True)
os.makedirs(path_to_H2_gas, exist_ok=True)
Poscar(h2_gas).write_file(f'{path_to_H2_gas}/POSCAR')

# Relax the bare surface (if it is not already relaxed) and the H2 reference
for folder in [surface_dir, path_to_H2_gas]:
    energy_lookup.ensure_relaxed(folder, model_load_path, relax_cell=False, device=device)   # -> <folder>/relaxation/CONTCAR
    print(f'{folder}: relaxed')

## Sample the adsorption sites
Atop, bridge and hollow sites come from a periodic Voronoi decomposition of the top layer; sites with equivalent local environments
are merged. Each H$_2$ site is generated with the molecule perpendicular and parallel to the surface. H configurations go in
`free-H` and H$_2$ configurations in `free-H2` (existing configurations are reused).

In [ ]:
for adsorbate, output_dir in [('H', path_to_free_H), ('H2', path_to_free_H2)]:
    config_dirs = adsorption.generate_inequivalent_hydrogen_sites(
        surface_dir=surface_dir,
        output_dir=output_dir,
        adsorption_height=1.5,
        z_tolerance=0.25,
        adsorbate=adsorbate,
        surface=surface_side,
    )

    print(f'{len(config_dirs)} {adsorbate} configurations in {output_dir}:')
    for config_dir in config_dirs:
        print('   ', os.path.basename(config_dir))

In [ ]:
# Relax every configuration with the ML potential (the cell is kept fixed so the lattice vectors stay matched)
relax_dirs = []
for path_to_adsorbate in [path_to_free_H, path_to_free_H2]:
    relax_dirs += [f'{path_to_adsorbate}/{conf}' for conf in sorted(os.listdir(path_to_adsorbate)) if conf[:4] == 'conf']

for folder in relax_dirs:
    energy_lookup.ensure_relaxed(folder, model_load_path, relax_cell=False, device=device)   # -> <folder>/relaxation/CONTCAR
    print(f'{folder}: relaxed')

In [ ]:
# Reference lattice vectors: all configurations are built on the surface cell
lattice_vectors = vasp_files.read_lattice_vectors(surface_dir)

adsorbate_dirs = {'H': path_to_free_H, 'H2': path_to_free_H2}

# Every energy entering the adsorption energy must be at the same level of theory
energy_lookup.check_energy_sources([surface_dir, path_to_H2_gas] + relax_dirs,
                         label='adsorption energy', model_load_path=model_load_path)

confs_data = {}
for adsorbate, path_to_adsorbate in adsorbate_dirs.items():
    confs_data[adsorbate] = {}
    for conf in sorted(os.listdir(path_to_adsorbate)):
        if conf[:4] == 'conf':
            path_to_conf = f'{path_to_adsorbate}/{conf}'
            energy               = energy_lookup.read_energy(path_to_conf, model_load_path=model_load_path, device=device)
            lattice_vectors_conf = vasp_files.read_lattice_vectors(path_to_conf)

            # Lattice vectors match; skip any configuration without readable structure data or energy.
            if lattice_vectors is None or lattice_vectors_conf is None or not np.isfinite(energy):
                print(f'Missing lattice vectors or energy for {adsorbate}/{conf}; skipping this configuration.')
                continue
            if np.allclose(lattice_vectors, lattice_vectors_conf):
                confs_data[adsorbate][conf] = energy
            else:
                print(f'Lattice vectors do not match for {adsorbate}/{conf}')

# Lowest-energy configuration for each adsorbate
min_conf   = {ads: min(data, key=data.get) for ads, data in confs_data.items() if data}
min_energy = {ads: confs_data[ads][conf]   for ads, conf in min_conf.items()}

for adsorbate in adsorbate_dirs:
    if adsorbate in min_conf:
        print(f'{adsorbate}: minimum energy {min_energy[adsorbate]} for {min_conf[adsorbate]}')
    else:
        print(f'No valid {adsorbate} configurations found in {adsorbate_dirs[adsorbate]}.')

In [ ]:
if not min_conf:
    print(f'Skipping plot for {surface}: no valid adsorption configuration was found.')
else:
    # Plot confs_data for each adsorbate
    fig, axes = plt.subplots(1, len(min_conf), figsize=(6*len(min_conf), 4), squeeze=False)
    for ax, adsorbate in zip(axes[0], min_conf):
        data = confs_data[adsorbate]
        sns.barplot(x=list(data.keys()), y=list(data.values()), ax=ax)
        ax.axhline(y=min_energy[adsorbate], color='r', linestyle='--',
                   label=f'Minimum energy: {min_energy[adsorbate]:2g} at {min_conf[adsorbate]}')
        ax.tick_params(axis='x', rotation=45)
        ax.set_ylim(min(data.values())-0.1, max(data.values())+0.1)
        ax.set_ylabel('Energy (eV)')
        ax.set_title(adsorbate)
        ax.legend(loc='best')

    fig.tight_layout()
    fig.savefig(f'{path_to_confs}/H-absorption.pdf', dpi=50)
    plt.show()

In [ ]:
if not min_conf:
    print(f'Skipping replica count for {surface}: no valid adsorption configuration was found.')
else:
    # Number of replicas of the surface in each lowest-energy configuration (1 when the configuration is
    # built on the surface cell; larger for adsorption supercells supplied from DFT)
    surface_volume = vasp_files.read_volume(surface_dir)
    n_replicas = {}
    for adsorbate, conf in min_conf.items():
        min_conf_volume = vasp_files.read_volume(f'{adsorbate_dirs[adsorbate]}/{conf}')
        ratio = min_conf_volume / surface_volume
        n_replicas[adsorbate] = np.round(ratio)
        if n_replicas[adsorbate] < 1 or not np.isclose(ratio, n_replicas[adsorbate], atol=1e-3):
            raise ValueError(f'{adsorbate}/{conf}: volume ratio {ratio:.3f} to the bare surface is not a positive integer.')

In [ ]:
if not min_conf:
    print(f'Skipping free-energy calculations for {surface}: no valid adsorption configuration was found.')
else:
    # Extract energies of the isolated H2 molecule and the bare surface
    hydrogen_molecule_energy = energy_lookup.read_energy(path_to_H2_gas, model_load_path=model_load_path, device=device)
    surface_energy           = energy_lookup.read_energy(surface_dir,    model_load_path=model_load_path, device=device)

In [ ]:
if not min_conf:
    print(f'Skipping adsorption-energy calculation for {surface}: no valid adsorption configuration was found.')
else:
    summary = {'surface': surface, 'surface_side': surface_side, 'model': model_load_path}

    # Dissociative adsorption (per H atom, referenced to 1/2 H2)
    if 'H' in min_conf:
        absorption_energy_atom = min_energy['H'] - (n_replicas['H']*surface_energy + hydrogen_molecule_energy/2)
        np.savetxt(f'{path_to_confs}/H_absorption_energy', [absorption_energy_atom])
        print(f"H adsorption energy (eV, per H atom, vs 1/2 H2): {absorption_energy_atom:.3f}  [{min_conf['H']}]")
        summary['H'] = {'conf': min_conf['H'], 'adsorption_energy_eV': float(absorption_energy_atom)}

    # Molecular adsorption of H2 (referenced to H2)
    if 'H2' in min_conf:
        absorption_energy_molecule = min_energy['H2'] - (n_replicas['H2']*surface_energy + hydrogen_molecule_energy)
        np.savetxt(f'{path_to_confs}/H2_absorption_energy', [absorption_energy_molecule])

        # Check whether the lowest-energy H2 configuration is still a molecule after the relaxation
        hh_distance = adsorption.get_hydrogen_pair_distance(f"{path_to_free_H2}/{min_conf['H2']}")
        mode = 'unknown' if hh_distance is None else ('molecular' if hh_distance < hh_molecular_threshold else 'dissociated')
        print(f"H2 adsorption energy (eV, per H2, vs H2): {absorption_energy_molecule:.3f}  [{min_conf['H2']}], "
              f"H-H distance {hh_distance if hh_distance is None else round(hh_distance, 2)} Å -> {mode}")
        if mode == 'dissociated':
            print('    The H2 configuration dissociated during the relaxation: this energy is for dissociative adsorption of a '
                  'whole H2 molecule (two H atoms), not physisorption. Compare it with 2x the per-atom H energy.')
        summary['H2'] = {'conf': min_conf['H2'], 'adsorption_energy_eV': float(absorption_energy_molecule),
                         'hh_distance_A': None if hh_distance is None else float(hh_distance), 'mode': mode}

    with open(f'{path_to_confs}/adsorption_summary.json', 'w') as json_file:
        json.dump(summary, json_file, indent=2)

# Finite-temperature adsorption free energy (optional)

At temperature $T$ the same expression is used with the free energies at that temperature:

\begin{equation}
    \Delta G_{ads}(T) = G_{conf}(T) - \left( n G_{surf}(T) + \tfrac{n_H}{2} G_{H_2}(T) \right), \qquad G(T) = U_0(T) + F_{vib}(T)
\end{equation}

* **Surface and configurations.** $G_{surf}$ and $G_{conf}$ come from NPT runs, `<folder>/md/<T>K`, exactly as for the slabs in
  `iterate-slabs` (per atom, times the atoms of the cell of the configuration). The bare surface needs its runs from `generate-slabs`
  (`finite_temperature = True`); every configuration, **all of them**, gets one run per temperature of the surface. The settings of
  the runs (length, thermostat, barostat, thermalisation time `equil_ps`, treatment of the cell) are read from the runs of the surface, so
  nothing is defined twice; only the time step and the frame spacing are shortened for hydrogen (the O-H stretch reaches ~450 meV).
  The runs start from the relaxed configuration, so that the H does not leave its site while thermalising.
* **H$_2$ reference.** $G_{H_2}$ is the free energy of the **molecular vibration only**: $G_{H_2} = E_0 + F_{vib}$, with $F_{vib}$ the
  quantum free energy of the stretching mode (zero-point energy included) at the harmonic frequency of the relaxed molecule, computed
  here with the ML-IAP (or given in `omega_h2_meV`, for instance from DFT). The translation and the rotation of the molecule are **not**
  included, and so neither is the pressure of the gas: their free energy is about $-0.4$ eV at 300 K and 1 bar, so $\Delta G_{ads}(T)$ is a
  *vibrational* free energy of adsorption, with the gas referenced at its vibrational level, and not the free energy of adsorption from
  H$_2$ gas at a given pressure. A further term is one entry (`extra_h2_terms`) of `adsorption.adsorption_free_energy`.
* **Table.** The contribution of every term of $G$ to $\Delta G_{ads}$ is in a column (`d_U0_eV`, `d_F_vib_eV`), and the flags report what
  the analysis of each run found: an H that desorbed, an H$_2$ that dissociated, an H that hops between sites (the harmonic spectrum
  is then not a good description of it), and the usual checks of the runs.

One NPT run is needed per configuration and temperature, so the cost grows with the number of configurations: use a GPU for the
ML-IAP or run them with VASP (put the `vasprun.xml` in the `md/<T>K` folder).

In [ ]:
# Finite temperature (optional): needs the md/<T>K folders of the bare surface (generate-slabs with finite_temperature = True)
finite_temperature = True
operating_T        = 300.0    # K, temperature of the free energies (it need not be an MD temperature, see iterate-slabs)
run_md             = True    # True: run the pending NPT runs here with the ML-IAP (slow on a CPU; use a GPU), or run them with VASP

# Hydrogen needs a shorter time step than the surface; the duration of the run in ps is kept
md_timestep    = 0.5          # fs
md_loginterval = 4            # a frame every md_loginterval * md_timestep = 2 fs (Nyquist frequency of the spectrum: 1 eV)

# Stretching energy of the isolated H2 (meV) for the reference: None -> computed with the ML-IAP from the relaxed molecule
omega_h2_meV = None

In [ ]:
from libraries import thermal

conf_folders = []
if finite_temperature:
    temperatures = thermal.md_temperatures(surface_dir)
    md_settings  = thermal.md_settings_of(surface_dir, timestep=md_timestep, loginterval=md_loginterval)
    if not temperatures:
        raise RuntimeError(f'no md/<T>K folder in {surface_dir}: generate them with generate-slabs (finite_temperature = True) '
                           f'for this surface first')
    for adsorbate, path_to_adsorbate in adsorbate_dirs.items():
        for conf in sorted(os.listdir(path_to_adsorbate)):
            if conf[:4] == 'conf':
                folder = f'{path_to_adsorbate}/{conf}'
                thermal.prepare_md_folders(folder, 'slab', temperatures, adsorbate=adsorbate, start_from='relaxed', **md_settings)
                conf_folders.append(folder)
    print(f'{len(conf_folders)} configuration(s) x {len(temperatures)} temperature(s): {temperatures} K; settings of the surface, '
          f'thermalisation {md_settings["equil_ps"]:g} ps, cell {md_settings["fix_cell"]!r}')
else:
    print('finite_temperature = False: only the 0 K adsorption energies are computed.')

In [ ]:
if finite_temperature:
    if run_md:
        thermal.run_pending_md_in([surface_dir] + conf_folders, model_load_path=model_load_path, device=device)

    md_status = thermal.md_status_of([('surface', surface_dir)] + [(f.replace(path_to_confs + '/', ''), f) for f in conf_folders])
    print(md_status['state'].value_counts().to_string())
    if (md_status['state'] != 'done').any():
        print('\nRuns that are not finished are left out (NaN) of the free-energy table:')
        display(md_status[md_status['state'] != 'done'])
        if not run_md:
            print('[note] they have not been run: set run_md = True to run them here, or run them with VASP.')

In [ ]:
if finite_temperature:
    thermal.check_md_consistency([surface_dir] + conf_folders, model_load_path)
    free_energy = adsorption.adsorption_free_energy(surface_dir, adsorbate_dirs, path_to_H2_gas, operating_T,
                                                    model_load_path=model_load_path, device=device, omega_h2_meV=omega_h2_meV)
    display(free_energy.drop(columns='flags'))

    reference = free_energy.attrs['h2_reference']
    print(f"H2 reference at {operating_T:g} K: G = {reference['G_eV']:.3f} eV per molecule = "
          + ' + '.join(f'{v:.3f} ({k})' for k, v in reference['terms'].items())
          + f"; stretch {reference['omega_meV']:.0f} meV ({reference['omega_source']}); not included: {reference['not_included']}")

    for _, row in free_energy[free_energy['flags'] != ''].iterrows():
        print(f"\n[flags] {row['adsorbate']}/{row['conf']}: {row['flags']}")

    print('\nLowest configuration of each adsorbate (level T: at the operating temperature):')
    display(adsorption.lowest_configurations(free_energy).drop(columns='flags'))

    free_energy.to_csv(f'{path_to_confs}/adsorption_free_energy_{operating_T:g}K.csv', index=False)

In [ ]:
if finite_temperature:
    valid = free_energy.dropna(subset=['dG_ads_T_eV']).copy()
    valid['configuration'] = valid['conf'] + valid['flags'].map(lambda f: '*' if f else '')
    adsorbates = list(valid['adsorbate'].unique())
    if not adsorbates:
        print('No configuration has a finished run: nothing to plot.')
    else:
        fig, axes = plt.subplots(1, len(adsorbates), figsize=(6 * len(adsorbates), 4), squeeze=False)
        for ax, adsorbate in zip(axes[0], adsorbates):
            data = valid[valid['adsorbate'] == adsorbate].melt(id_vars='configuration', value_vars=['dG_ads_0K_eV', 'dG_ads_T_eV'],
                                                               var_name='level', value_name='dG_ads')
            data['level'] = data['level'].map({'dG_ads_0K_eV': '0 K', 'dG_ads_T_eV': f'{operating_T:g} K'})
            sns.barplot(data=data, x='configuration', y='dG_ads', hue='level', ax=ax)
            ax.tick_params(axis='x', rotation=45)
            ax.set_ylabel(r'$\Delta G_{ads}$ (eV)')
            ax.set_title(f'{adsorbate} (* has flags)')
        fig.tight_layout()
        fig.savefig(f'{path_to_confs}/adsorption_free_energy_{operating_T:g}K.pdf', dpi=50)
        plt.show()